## WorldMove Data EDA

### i. Libraries

In [20]:
import pandas as pd
import numpy as np
import seaborn as sns
import scipy as sp
import json
import matplotlib.pyplot as plt
import geopandas as gpd
import folium
from shapely.geometry import Polygon, MultiPolygon, Point
from branca.colormap import LinearColormap
import warnings  # Optional: to suppress if you want
import zipfile #opening zips


### ii. Overview

WorldMove is a large-scale synthetic mobility dataset covering over 1,600 cities across 179 countries and 6 continents. The dataset uses a diffusion-based generative model to create realistic urban mobility patterns from publicly available data sources.

Links

- **Official Website**: [WorldMove Dataset Portal](https://www.fiblab.net/worldmove/)
- **GitHub Repository**: [https://github.com/tsinghua-fib-lab/WorldMove](https://github.com/tsinghua-fib-lab/WorldMove)
- **Research Paper**: [arXiv:2504.10506](https://arxiv.org/abs/2504.10506)


### iii. Descriptive Analysis

#### a. Grid-cell coordinates

In [3]:
# load grid coordinates from JSON file
coord_file = '/home/dataopske/Desktop/jav/data/raw/worldmove/gridcell_coordinates.json'

with open(coord_file, 'r') as f:
    coordinates = json.load(f)

**File Format**: `.json`  
**Example**: `1098_KE_Nairobi.json`  
**Structure**:
```json
{
  "0": [longitude, latitude],
  "1": [longitude, latitude],
  ...
  "1439": [longitude, latitude]
}
```
**Description**: Centers of 1km x 1km grid cells covering Nairobi (1,440 cells total)  
**Coordinate System**: WGS84 (EPSG:4326)

In [4]:
# convert to DataFrame the coord file into a dataframe
coord_df = pd.DataFrame([
    {'cell_id': int(k), 'lon': v[0], 'lat': v[1]}
    for k, v in coordinates.items()
])

coord_df.shape

(1440, 3)

Centers of 1km x 1km grid cells covering Nairobi (1,440 cells total) 

#### b. Grid-cell Population

In [5]:
# load population data
pop_file = '/home/dataopske/Desktop/jav/data/raw/worldmove/grid-cell_population.npy'

population = np.load(pop_file)

# population summary
print(f"  Population grid shape: {population.shape}")
print(f"  Total population: {population.sum():,.0f}")
print(f"  Max per cell: {population.min():,.0f}")
print(f"  Max per cell: {population.max():,.0f}")
print(f"  Mean per cell: {population.mean():.0f}")

  Population grid shape: (30, 48)
  Total population: 20,754,780
  Max per cell: 23
  Max per cell: 280,940
  Mean per cell: 14413


Since WorldMove generates synthetic trajectories, the `20M` might represent the `number of simulated agents they generated for realistic mobility patterns, not actual people`. The model might have intentionally oversample to ensure statistical coverage of all mobility patterns.

Since the data covers the Nairobi Metropolitan area. I will normalise the data to `approx 5.7M` as per the latest `KNBS estimate`

This notebook cell processes a gridded population dataset (e.g., from WorldPop) to calibrate estimates specifically for the Nairobi metropolitan area. It uses a `point-in-polygon test` to mask grid cells inside Nairobi's boundaries, then applies a uniform scaling factor to match a known total population of `4.8 million`. The result is a corrected population DataFrame with spatial flags, suitable for downstream urban modeling (e.g., mobility or demographic analysis).

In [6]:
# Point-in-polygon function (ray-casting)
def point_in_polygon(x, y, poly):
    n = len(poly)
    inside = False
    p1x, p1y = poly[0]
    for i in range(n + 1):
        p2x, p2y = poly[i % n]
        if y > min(p1y, p2y):
            if y <= max(p1y, p2y):
                if x <= max(p1x, p2x):
                    if p1y != p2y:
                        xinters = (y - p1y) * (p2x - p1x) / (p2y - p1y) + p1x
                    if p1x == p2x or x <= xinters:
                        inside = not inside
        p1x, p1y = p2x, p2y
    return inside

# Load Nairobi JSON (robust parsing)
nairobi_file = '/home/dataopske/Desktop/jav/data/raw/supportdata/nairobi.json'
try:
    with open(nairobi_file, 'r') as f:
        nairobi_data = json.load(f)
    
    # Auto-detect structure
    if 'features' in nairobi_data and nairobi_data['features']:
        feature = nairobi_data['features'][0]  # FeatureCollection
    else:
        feature = nairobi_data  # Single Feature or raw
    
    geom = feature['geometry']
    poly_type = geom['type']
    
    if poly_type == 'Polygon':
        poly_coords = geom['coordinates'][0]  # Exterior ring
    elif poly_type == 'MultiPolygon':
        poly_coords = geom['coordinates'][0][0]  # First polygon's exterior
    elif poly_type == 'GeometryCollection':
        # Fallback to first Polygon
        for child in geom['geometries']:
            if child['type'] == 'Polygon':
                poly_coords = child['coordinates'][0]
                break
        else:
            raise ValueError("No Polygon in GeometryCollection")
    else:
        raise ValueError(f"Unsupported geometry type: {poly_type}")
    
    poly = np.array(poly_coords)  # [lon, lat] assumed
    print(f"Loaded polygon: {poly_type} with {len(poly)} points")
    
except Exception as e:
    print(f"JSON error: {e}. Falling back to bounds mask.")
    # Fallback: Rough Nairobi box (lon 36.75-36.95, lat -1.4 to -1.2)
    poly = np.array([[36.75, -1.4], [36.95, -1.4], [36.95, -1.2], [36.75, -1.2], [36.75, -1.4]])  # Square approx

# Load raw data
# pop_file = '/home/dataopske/Desktop/jav/data/raw/worldmove/grid-cell_population.npy'
population_raw = np.load(pop_file)
pop_flat_raw = population_raw.flatten()

# Generate centroids if missing
if 'lat' not in coord_df.columns or 'lon' not in coord_df.columns:
    rows, cols = population_raw.shape
    lat_min, lat_max, lon_min, lon_max = -1.47, -1.0, 36.65, 37.0
    lat_centers = np.linspace(lat_max - 0.0045, lat_min + 0.0045, rows)
    lon_centers = np.linspace(lon_min + 0.0045, lon_max - 0.0045, cols)
    lat_grid, lon_grid = np.meshgrid(lat_centers, lon_centers, indexing='ij')
    lats_1d, lons_1d = lat_grid.flatten(), lon_grid.flatten()
    coord_df = pd.DataFrame({'lat': lats_1d, 'lon': lons_1d})

# Compute inside_nairobi mask
inside_nairobi = [point_in_polygon(lon, lat, poly) for lon, lat in zip(coord_df['lon'], coord_df['lat'])]
coord_df['inside_nairobi'] = inside_nairobi
n_inside = coord_df['inside_nairobi'].sum()
print(f"Mask: {n_inside} / {len(coord_df)} cells inside")

# Step 1: Assign raw pop
coord_df['population_raw'] = pop_flat_raw[:len(coord_df)]

# Step 2: Calculate raw population INSIDE
raw_inside_nairobi = coord_df.loc[coord_df['inside_nairobi'], 'population_raw'].sum()

# Step 3: Known pop
known_nairobi_pop = 4_800_000

# Step 4: Factor
normalization_factor = raw_inside_nairobi / known_nairobi_pop

print(f"Raw inside: {raw_inside_nairobi:,.0f}")
print(f"Known: {known_nairobi_pop:,.0f}")
print(f"Factor: {normalization_factor:.4f}x")

# Step 5: Apply to entire
coord_df['population_corrected'] = coord_df['population_raw'] / normalization_factor

# Results
total_corrected = coord_df['population_corrected'].sum()
inside_corrected = coord_df.loc[coord_df['inside_nairobi'], 'population_corrected'].sum()
outside_corrected = total_corrected - inside_corrected

print(f"\n✓ Corrected:")
print(f"  Inside: {inside_corrected:,.0f}")
print(f"  Outside: {outside_corrected:,.0f}")
print(f"  Total: {total_corrected:,.0f}")

# # Viz
# plt.scatter(coord_df['population_raw'], coord_df['population_corrected'], alpha=0.6)
# plt.plot([0, coord_df['population_raw'].max()], [0, coord_df['population_raw'].max() / normalization_factor], 'r--')
# plt.xlabel('Raw'); plt.ylabel('Corrected')
# plt.title('Scaling Check')
# plt.savefig('scaling_viz.png')
# plt.show()

coord_df.to_csv('/home/dataopske/Desktop/jav/data/processed/coord_df_corrected.csv', index=False)
print("\nExported!")

JSON error: 'geometry'. Falling back to bounds mask.
Mask: 505 / 1440 cells inside
Raw inside: 15,890,350
Known: 4,800,000
Factor: 3.3105x

✓ Corrected:
  Inside: 4,800,000
  Outside: 1,469,399
  Total: 6,269,399

Exported!


In [9]:
# the new column with `population_corrected`
coord_df.head(3)

,cell_id,lon,lat,inside_nairobi,population_raw,population_corrected
0,0,36.668782,-1.165734,False,4899.411133,1479.965753
1,1,36.675437,-1.167976,False,2851.202393,861.263075
2,2,36.689270,-1.165173,False,1454.953857,439.498099


The new df is now `coord_df` which has `Grid-cell coordinates` now has the normalised population data mapped (1*1km). 

In [18]:
# Optional: Suppress the specific warning (use sparingly)
warnings.filterwarnings('ignore', message='Geometry is in a geographic CRS')

# Load Nairobi boundary (GeoJSON)
nairobi_boundary_file = '/home/dataopske/Desktop/jav/data/raw/supportdata/nairobi.json'
nairobi_gdf = gpd.read_file(nairobi_boundary_file)
# Ensure CRS is WGS84 (lat/lon)
nairobi_gdf = nairobi_gdf.to_crs(epsg=4326)
# Extract polygon geometry safely
geom = nairobi_gdf.geometry.iloc[0]
# Handle cases: GeometryCollection, MultiPolygon, or Polygon
if geom.geom_type == 'GeometryCollection':
    # Filter for the first polygon in the collection
    polygons = [g for g in geom.geoms if g.geom_type in ['Polygon', 'MultiPolygon']]
    if polygons:
        geom = polygons[0]
    else:
        raise ValueError("No polygon geometry found in the collection!")
if geom.geom_type == 'MultiPolygon':
    # Use the largest polygon by area (common for administrative boundaries)
    geom = max(geom.geoms, key=lambda g: g.area)
# Now safely extract coordinates
boundary_coords = np.array(geom.exterior.coords)
boundary_flipped = [[y, x] for x, y in boundary_coords]  # Folium expects [lat, lon]

# Load or assume coord_df with population data (from previous processing)
# Uncomment if not in memory:
# coord_df = pd.read_csv('coord_df_corrected.csv')
# Ensure required columns exist
required_cols = ['lat', 'lon', 'population_corrected', 'population_raw']
if not all(col in coord_df.columns for col in required_cols):
    raise ValueError(f"coord_df missing columns: {set(required_cols) - set(coord_df.columns)}")

# Compute dynamic inside mask using the loaded geom (vectorized for efficiency)
points_gdf = gpd.GeoDataFrame(
    coord_df, 
    geometry=gpd.points_from_xy(coord_df.lon, coord_df.lat), 
    crs='EPSG:4326'
)
inside_mask = points_gdf.within(geom)
coord_df['inside_nairobi_dynamic'] = inside_mask  # Add for reference

# FIXED: Compute map center from boundary centroid (project for accuracy)
# Project to UTM 37N (meters-based, accurate for Kenya)
nairobi_proj = nairobi_gdf.to_crs('EPSG:32637')
cent_proj = nairobi_proj.geometry.centroid.iloc[0]  # No warning here!
center_lon = cent_proj.x  # Easting in meters -> but wait, we need lon/lat
center_lat = cent_proj.y  # Northing in meters

# Transform back to 4326 for map-friendly degrees
nairobi_back = nairobi_proj.to_crs('EPSG:4326')
center_lon = nairobi_back.geometry.centroid.x.iloc[0]
center_lat = nairobi_back.geometry.centroid.y.iloc[0]
print(f"Accurate centroid: [{center_lat:.6f}, {center_lon:.6f}]")  # Extra precision shown

# === Plot on Folium ===
m = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles="cartodbpositron")

# Add Nairobi boundary polygon
folium.Polygon(
    locations=boundary_flipped,
    color="blue",
    weight=2,
    fill=True,
    fill_opacity=0.15,
    popup="Nairobi Boundary"
).add_to(m)

# Create color map for corrected population
pop_max = coord_df['population_corrected'].max()
pop_colormap = LinearColormap(
    colors=['#ffffcc', '#ffeda0', '#fed976', '#feb24c', '#fd8d3c', '#fc4e2a', '#e31a1c', '#bd0026'],
    vmin=0,
    vmax=pop_max,
    caption='Corrected Population per Cell (~km²)'
)

# Group for population markers (corrected)
pop_group = folium.FeatureGroup(name='Corrected Population', show=True)

# Add population circles (only for cells with pop > 0)
for idx, row in coord_df.iterrows():
    if row['population_corrected'] > 0:
        # Radius scaled by sqrt for area representation; adjust 0.05 for visibility
        radius = np.sqrt(row['population_corrected']) * 0.05
        color = pop_colormap(row['population_corrected'])
        is_inside = row['inside_nairobi_dynamic']
        
        # Base popup content
        popup_html = f"<b>Cell {idx}</b><br>Corrected Pop: {row['population_corrected']:,.0f}<br>Raw Pop: {row['population_raw']:,.0f}<br>Density: ~{row['population_corrected']:,.0f} people/km²<br>Location: [{row['lat']:.4f}, {row['lon']:.4f}]"
        if is_inside:
            popup_html += "<br><b>Inside Nairobi</b>"
        else:
            popup_html += "<br><i>Outside Nairobi</i>"
        
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=radius,
            zoom_start=11,
            popup=folium.Popup(popup_html, max_width=250),
            tooltip=f"Cell {idx}: {row['population_corrected']:,.0f} people (Inside: {is_inside})",
            color=color if is_inside else '#808080',  # Gray outline for outside
            fill=True,
            fillColor=color if is_inside else '#d3d3d3',  # Gray fill for outside
            fillOpacity=0.6 if is_inside else 0.3,
            weight=1
        ).add_to(pop_group)

pop_group.add_to(m)

# Add tile layers for variety
folium.TileLayer('OpenStreetMap', name='Default').add_to(m)

# folium.TileLayer('cartodbdark_matter', name='Dark Map').add_to(m)

# Add legend and controls
pop_colormap.add_to(m)
folium.LayerControl().add_to(m)

# Display inline in Jupyter
m

Accurate centroid: [-1.298169, 36.859697]


#### c. Wards: from `/data/raw/kenyawards` dir

In order to aggregate the population into wards for easier visualisation

In [19]:
# Open the ZIP file in read mode ('r')
zip_path = '/home/dataopske/Desktop/jav/data/raw/kenyawards/kenya_wards.zip'  # Replace with your ZIP file path
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    file_list = zip_ref.namelist()  # Returns a list of file paths (including subdirs)
    print(file_list)  # e.g., ['data.csv', 'folder/image.jpg']
    pass

['Kenya_Wards/', 'Kenya_Wards/kenya_wards.cpg', 'Kenya_Wards/kenya_wards.dbf', 'Kenya_Wards/kenya_wards.prj', 'Kenya_Wards/kenya_wards.qpj', 'Kenya_Wards/kenya_wards.shp', 'Kenya_Wards/kenya_wards.shx']


The zip contains multiple files, we're interested in the `Kenya_Wards/kenya_wards.shp`

In [22]:
# Read the specific shapefile inside the zip
wards_gdf = gpd.read_file(f"zip://{zip_path}!Kenya_Wards/kenya_wards.shp")

# Show first few rows
wards_gdf.head(3)
# wards_gdf.crs

,gid,pop2009,county,subcounty,ward,uid,scuid,cuid,geometry
0,241,17431.0,ISIOLO,Isiolo Sub County,WABERA,rIdiIpv9fBt,I2LYLqKU6AW,bzOfj0iwfDH,"POLYGON ((37.59968 0.40029, 37.59976 0.40004, ..."
1,1455,18755.0,Migori,Rongo Sub County,North Kamagambo Ward,QC41mItjIzF,fT37q3rXQ35,fVra3Pwta0Q,"POLYGON ((34.59938 -0.65054, 34.60006 -0.65069..."
2,1456,27756.0,Migori,Rongo Sub County,Central Kamagambo Ward,M8rGveWTIMm,fT37q3rXQ35,fVra3Pwta0Q,"POLYGON ((34.61175 -0.73357, 34.61183 -0.73357..."


This `wards_gdf` file contains all the wards in Kenya. We shall filter for `Nairobi` county.

In [23]:
# filter wards by nairobi county into a new df
wards_nbo =  wards_gdf.loc[wards_gdf['county'] == 'Nairobi']

In [25]:
print(wards_nbo.shape)
wards_nbo.head(3)

(85, 9)


,gid,pop2009,county,subcounty,ward,uid,scuid,cuid,geometry
593,2044,55158.0,Nairobi,Ruaraka Sub County,Mathare North Ward,dOQTHZsuaIb,Cc8uEFkzfVf,jkG3zaihdSs,"POLYGON ((36.87653 -1.2521, 36.87618 -1.25223,..."
596,2047,70641.0,Nairobi,Embakasi South Sub County,Imara Daima Ward,Nt7PPe0Vdou,aDp1odOWYC1,jkG3zaihdSs,"POLYGON ((36.87934 -1.31037, 36.87934 -1.31037..."
920,2008,38384.0,Nairobi,Westlands Sub County,Parklands/highridge Ward,QhDd2LAuXAF,f1T0Ltob8VQ,jkG3zaihdSs,"POLYGON ((36.80712 -1.24986, 36.80714 -1.24988..."


> Now we only have the `85 wards` in Nairobi county for our EDA

In [26]:
# save filtered_df to processed dir
wards_nbo.to_csv('/home/dataopske/Desktop/jav/data/processed/wards_nbo.csv', index=False)

visualise the different wards in Nairobi

In [ ]:
# Your GeoDataFrame (already loaded)
zip_path = '/home/dataopske/Desktop/jav/data/raw/kenyawards/kenya_wards.zip'
wards_gdf = gpd.read_file(f"zip://{zip_path}!Kenya_Wards/kenya_wards.shp")

# Make sure it's in WGS84 (latitude/longitude) for Folium
wards_nbo = wards_nbo.to_crs(epsg=4326)

# Center the map (Kenya)
m = folium.Map(location=[0.1, 37.9], zoom_start=6, tiles='CartoDB positron')

# Add the wards GeoDataFrame
folium.GeoJson(
    wards_nbo,
    name='Nairobi Wards',
    style_function=lambda feature: {
        'fillColor': '#3186cc',
        'color': 'black',
        'weight': 0.4,
        'fillOpacity': 0.4,
    },
    tooltip=folium.features.GeoJsonTooltip(fields=['county', 'subcounty', 'ward'])
).add_to(m)

# Add layer controla
folium.LayerControl().add_to(m)

m

#### d. Grid-cell coordinates + Wards + Population

In [29]:
# convert our coord_df into a geodataframe
from shapely.geometry import Point

# Create geometry column from lon/lat
coord_gdf = gpd.GeoDataFrame(
    coord_df,
    geometry=gpd.points_from_xy(coord_df['lon'], coord_df['lat']),
    crs='EPSG:4326'
)
coord_gdf = coord_gdf.loc[coord_gdf['inside_nairobi_dynamic'] == True] #only include the populations inside Nairobi

In [30]:
# project wards_nbo to crs
wards_nbo = wards_nbo.to_crs(epsg=4326)

1. Spatial join to assign each grid to a ward

In [32]:
# join coord_gdf with wards_nbo
coord_with_wards = gpd.sjoin(
    coord_gdf,
    wards_nbo[['ward', 'subcounty', 'geometry']], 
    how='left',
    predicate='within'   # checks if point lies inside a ward polygon
)

`coord_with_wards` is a combination of `Grid-cell coordinates`, `population data` and `wards data`. 

In [33]:
coord_with_wards.head(3)

,cell_id,lon,lat,inside_nairobi,population_raw,population_corrected,inside_nairobi_dynamic,geometry,index_right,ward,subcounty
26,26,36.902065,-1.164995,False,9894.172852,2988.734070,True,POINT (36.90206 -1.16499),1197.0,Kahawa West,Roysambu Sub County
28,28,36.917433,-1.167603,False,5419.624023,1637.106528,True,POINT (36.91743 -1.1676),1197.0,Kahawa West,Roysambu Sub County
73,73,36.897392,-1.171370,False,29171.931641,8811.969155,True,POINT (36.89739 -1.17137),1197.0,Kahawa West,Roysambu Sub County


In [42]:
coord_with_wards['population_corrected'].sum()

np.float64(5354517.076610196)

In [43]:
# Save this data to processed dir
coord_with_wards.to_csv('/home/dataopske/Desktop/jav/data/processed/coord_with_wards.csv', index=False)

2. Aggregate population per ward

In [44]:
ward_pop = (
    coord_with_wards
    .groupby('ward', as_index=False)['population_corrected']
    .sum()
    .rename(columns={'population_corrected': 'population'})
)

wards_nbo = wards_nbo.merge(ward_pop, on='ward', how='left')
wards_nbo['population'] = wards_nbo['population'].fillna(0)

In [37]:
wards_nbo.to_csv('/home/dataopske/Desktop/jav/data/processed/wards_nbo_pop.csv', index=False)

In [45]:
wards_nbo.head(3)

,gid,pop2009,county,subcounty,ward,uid,scuid,cuid,geometry,population_x,population_y,population
0,2044,55158.0,Nairobi,Ruaraka Sub County,Mathare North Ward,dOQTHZsuaIb,Cc8uEFkzfVf,jkG3zaihdSs,"POLYGON ((36.87653 -1.2521, 36.87618 -1.25223,...",74177.992829,74177.992829,74177.992829
1,2047,70641.0,Nairobi,Embakasi South Sub County,Imara Daima Ward,Nt7PPe0Vdou,aDp1odOWYC1,jkG3zaihdSs,"POLYGON ((36.87934 -1.31037, 36.87934 -1.31037...",92557.211478,92557.211478,92557.211478
2,2008,38384.0,Nairobi,Westlands Sub County,Parklands/highridge Ward,QhDd2LAuXAF,f1T0Ltob8VQ,jkG3zaihdSs,"POLYGON ((36.80712 -1.24986, 36.80714 -1.24988...",69612.843253,69612.843253,69612.843253


In [48]:
from branca.colormap import LinearColormap
import matplotlib.cm as cm
import matplotlib.colors as colors

# Choose a Matplotlib colormap ('viridis', 'inferno', 'plasma', 'magma', etc.)
cmap = cm.get_cmap('inferno', 256)  # change to 'viridis' if you prefer cooler tones
# cmap = matplotlib.colormaps.get_cmap('inferno', 256)

# Convert Matplotlib colormap to list of hex colors
color_list = [colors.rgb2hex(cmap(i)) for i in range(cmap.N)]

# Create a Folium LinearColormap
colormap = LinearColormap(
    colors=color_list,
    vmin=wards_nbo['pop_density'].min(),
    vmax=wards_nbo['pop_density'].max(),
    caption='Population Density (people/km²)'
)

# Create Folium map centered on Nairobi
m = folium.Map(location=[-1.29, 36.82], zoom_start=11, tiles='CartoDB positron')

# Add ward polygons colored by population density
folium.GeoJson(
    wards_nbo,
    style_function=lambda feature: {
        'fillColor': colormap(feature['properties']['pop_density']),
        'color': 'black',
        'weight': 0.5,
        'fillOpacity': 0.8
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['ward', 'subcounty', 'population', 'area_km2', 'pop_density'],
        aliases=['Ward', 'Subcounty', 'Population', 'Area (km²)', 'Density (/km²)'],
        localize=True
    )
).add_to(m)
# Add tile layers for variety
folium.TileLayer('OpenStreetMap', name='Default').add_to(m)
folium.TileLayer('cartodbdark_matter', name='Dark Map').add_to(m)

# Add legend and controls
pop_colormap.add_to(m)
folium.LayerControl().add_to(m)

colormap.add_to(m)
m


/tmp/ipykernel_13332/2384975653.py:6: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap('inferno', 256)  # change to 'viridis' if you prefer cooler tones


3. Calculate the geometric area, and the population density

In [46]:
import geopandas as gpd

# Convert to a projected CRS in meters (for accurate area calc)
wards_nbo = wards_nbo.to_crs(epsg=32737)  # UTM Zone 37S — covers Nairobi region well

# Compute area in km²
wards_nbo['area_km2'] = wards_nbo.geometry.area / 1_000_000

# Compute population density (people per km²)
wards_nbo['pop_density'] = wards_nbo['population'] / wards_nbo['area_km2']

# Convert back to WGS84 for Folium
wards_nbo = wards_nbo.to_crs(epsg=4326)


4. plot of area(km2), population and population density

In [47]:
import folium
from branca.colormap import LinearColormap

colormap = LinearColormap(
    colors=['#fff5f0', '#fcbba1', '#fc9272', '#fb6a4a', '#de2d26', '#a50f15'],
    vmin=wards_nbo['pop_density'].min(),
    vmax=wards_nbo['pop_density'].max(),
    caption='Population Density (people/km²)'
)


# Create map centered on Nairobi
m = folium.Map(location=[-1.29, 36.82], zoom_start=11, tiles='CartoDB positron')

# Add ward polygons colored by density
folium.GeoJson(
    wards_nbo,
    style_function=lambda feature: {
        'fillColor': colormap(feature['properties']['pop_density']),
        'color': 'black',
        'weight': 0.5,
        'fillOpacity': 0.7,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['ward', 'subcounty', 'population', 'area_km2', 'pop_density'],
        aliases=['Ward', 'Subcounty', 'Population', 'Area (km²)', 'Pop Density (/km²)'],
        localize=True
    ),
    name='Population Density'
).add_to(m)

# Add color legend
colormap.add_to(m)

# Add control
folium.LayerControl().add_to(m)

m